# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [35]:
# ML-05 Section 1: Method choice and setup

import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

RANDOM_STATE = 42

# Load Hugging Face token
hf_token = userdata.get("HF_TOKEN")

print("Token loaded:", "YES" if hf_token else "NO")

# Create DuckDB connection
con = duckdb.connect()

# Load warehouse files
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token
)

fact_march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

fact_april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=hf_token
)

print("Warehouse files loaded successfully.")

Token loaded: YES
Warehouse files loaded successfully.


In [36]:
# Build March 2026 feature table

features = con.execute(f"""
    SELECT
        dc.content_hash_id,
        dc.client_hash_id,
        dc.search_volume,
        dc.competition,
        dc.backlinks,
        dc.word_count,
        AVG(f.gsc_avg_position) AS avg_position_march
    FROM '{dim_content_path}' dc
    JOIN '{fact_march_path}' f
        ON dc.content_hash_id = f.content_hash_id
        AND dc.client_hash_id = f.client_hash_id
    WHERE dc.is_deleted IS FALSE
    GROUP BY
        dc.content_hash_id,
        dc.client_hash_id,
        dc.search_volume,
        dc.competition,
        dc.backlinks,
        dc.word_count
""").df()

print("Feature rows:", len(features))
print("Feature columns:", features.columns.tolist())

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 324947
Feature columns: ['content_hash_id', 'client_hash_id', 'search_volume', 'competition', 'backlinks', 'word_count', 'avg_position_march']


,content_hash_id,client_hash_id,search_volume,competition,backlinks,word_count,avg_position_march
0,content_05597932fe4da067,client_73cda7b4e4f265ea,10,1.00,<NA>,<NA>,2.714744
1,content_05434271b257bb68,client_73cda7b4e4f265ea,10,0.00,<NA>,<NA>,6.320337
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,50,0.06,<NA>,2475,4.459107
3,content_22610b0934f8825e,client_73cda7b4e4f265ea,110,0.37,<NA>,<NA>,12.791667
4,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,40,0.89,<NA>,<NA>,9.445635


In [37]:
# Create future outcome from April 2026

future_outcome = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS future_clicks,
        SUM(gsc_impressions) AS future_impressions
    FROM '{fact_april_path}'
    GROUP BY
        content_hash_id,
        client_hash_id
""").df()

print("Future outcome rows:", len(future_outcome))
print("Future outcome columns:", future_outcome.columns.tolist())

future_outcome.head()

Future outcome rows: 362172
Future outcome columns: ['content_hash_id', 'client_hash_id', 'future_clicks', 'future_impressions']


,content_hash_id,client_hash_id,future_clicks,future_impressions
0,content_76c1f31e2b38f054,client_62f4a7e64f5e0096,1.0,249.0
1,content_35d979572550dd7f,client_62f4a7e64f5e0096,0.0,1180.0
2,content_ffc5ab4b34aab1f8,client_62f4a7e64f5e0096,0.0,281.0
3,content_9739856fc83dc1ca,client_62f4a7e64f5e0096,0.0,722.0
4,content_d47ba5533f9c8573,client_62f4a7e64f5e0096,0.0,0.0


In [38]:
# Inspect the future outcome distribution before defining the label

print("Future clicks summary:")
print(future_outcome["future_clicks"].describe())

print("\nFuture impressions summary:")
print(future_outcome["future_impressions"].describe())

print("\nFuture clicks percentiles:")
print(
    future_outcome["future_clicks"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
)

Future clicks summary:
count    362172.000000
mean          2.319912
std          20.926011
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        7434.000000
Name: future_clicks, dtype: float64

Future impressions summary:
count    362172.000000
mean        806.432353
std        4240.388558
min           0.000000
25%           0.000000
50%           2.000000
75%         179.000000
max      799358.000000
Name: future_impressions, dtype: float64

Future clicks percentiles:
0.50     0.0
0.75     0.0
0.90     3.0
0.95     9.0
0.99    52.0
Name: future_clicks, dtype: float64


In [39]:
# Define the future success label
# Positive = at least 3 clicks in April 2026

model_data = features.merge(
    future_outcome,
    on=["content_hash_id", "client_hash_id"],
    how="inner"
)

model_data["target"] = (
    model_data["future_clicks"] >= 3
).astype(int)

print("Model rows:", len(model_data))
print("\nTarget distribution:")
print(model_data["target"].value_counts())

print("\nTarget rate:")
print(model_data["target"].mean())

Model rows: 324946

Target distribution:
target
0    290491
1     34455
Name: count, dtype: int64

Target rate:
0.10603300240655371


In [40]:
# Select leakage-safe features

feature_cols = [
    "search_volume",
    "competition",
    "backlinks",
    "word_count",
    "avg_position_march"
]

X = model_data[feature_cols].copy()
y = model_data["target"].copy()

# Convert numeric columns safely
X = X.apply(pd.to_numeric, errors="coerce")

# Fill missing values using training-safe simple defaults
X = X.fillna(0)

print("Features selected:", feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive rate:", y.mean())
print("\nMissing values:")
print(X.isna().sum())

Features selected: ['search_volume', 'competition', 'backlinks', 'word_count', 'avg_position_march']
X shape: (324946, 5)
y shape: (324946,)
Positive rate: 0.10603300240655371

Missing values:
search_volume         0
competition           0
backlinks             0
word_count            0
avg_position_march    0
dtype: int64


In [41]:
# Time-aware validation setup
# Features come from March; April is the future outcome.
# We keep a fixed holdout set for honest evaluation.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

Training rows: 259956
Test rows: 64990

Training positive rate: 0.10603332871716752
Test positive rate: 0.10603169718418218


In [42]:
# Baseline: rank pages using March average position
# Lower average position = better ranking performance.

from sklearn.metrics import average_precision_score

baseline_score = -X_test["avg_position_march"].values

# Precision@K helper
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1]
    top_k = order[:k]
    return y_true.iloc[top_k].mean()

# Evaluate baseline
baseline_p10 = precision_at_k(y_test.reset_index(drop=True),
                              baseline_score,
                              10)

baseline_p50 = precision_at_k(y_test.reset_index(drop=True),
                               baseline_score,
                               50)

baseline_p100 = precision_at_k(y_test.reset_index(drop=True),
                                baseline_score,
                                100)

baseline_pr_auc = average_precision_score(
    y_test,
    baseline_score
)

print("Baseline: March average-position ranking")
print("Precision@10 :", round(baseline_p10, 4))
print("Precision@50 :", round(baseline_p50, 4))
print("Precision@100:", round(baseline_p100, 4))
print("PR-AUC       :", round(baseline_pr_auc, 4))

Baseline: March average-position ranking
Precision@10 : 0.0
Precision@50 : 0.0
Precision@100: 0.0
PR-AUC       : 0.0745


In [43]:
# ML ranking model: Logistic Regression

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

# Probability of positive outcome
model_scores = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Test predictions:", len(model_scores))

Model trained successfully.
Test predictions: 64990


In [44]:
# Evaluate the ML ranking model

model_p10 = precision_at_k(
    y_test.reset_index(drop=True),
    model_scores,
    10
)

model_p50 = precision_at_k(
    y_test.reset_index(drop=True),
    model_scores,
    50
)

model_p100 = precision_at_k(
    y_test.reset_index(drop=True),
    model_scores,
    100
)

model_pr_auc = average_precision_score(
    y_test,
    model_scores
)

print("ML Ranking Model")
print("Precision@10 :", round(model_p10, 4))
print("Precision@50 :", round(model_p50, 4))
print("Precision@100:", round(model_p100, 4))
print("PR-AUC       :", round(model_pr_auc, 4))

ML Ranking Model
Precision@10 : 0.2
Precision@50 : 0.24
Precision@100: 0.21
PR-AUC       : 0.1718


In [45]:
# Compare baseline and ML model

comparison = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Precision@50",
        "Precision@100",
        "PR-AUC"
    ],
    "Baseline": [
        baseline_p10,
        baseline_p50,
        baseline_p100,
        baseline_pr_auc
    ],
    "ML Model": [
        model_p10,
        model_p50,
        model_p100,
        model_pr_auc
    ]
})

comparison["Absolute improvement"] = (
    comparison["ML Model"] - comparison["Baseline"]
)

comparison["Relative improvement %"] = (
    (comparison["ML Model"] - comparison["Baseline"])
    / comparison["Baseline"]
    * 100
)

comparison.round(4)

,Metric,Baseline,ML Model,Absolute improvement,Relative improvement %
0,Precision@10,0.0000,0.2000,0.2000,inf
1,Precision@50,0.0000,0.2400,0.2400,inf
2,Precision@100,0.0000,0.2100,0.2100,inf
3,PR-AUC,0.0745,0.1718,0.0972,130.4022


In [46]:
# Inspect model coefficients

coefficients = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.named_steps["classifier"].coef_[0]
})

coefficients["Absolute importance"] = coefficients["Coefficient"].abs()

coefficients = coefficients.sort_values(
    "Absolute importance",
    ascending=False
)

print("Feature contribution to the ranking model:")
coefficients.round(4)

Feature contribution to the ranking model:


,Feature,Coefficient,Absolute importance
3,word_count,0.8598,0.8598
0,search_volume,-0.1451,0.1451
2,backlinks,-0.0315,0.0315
1,competition,-0.0258,0.0258
4,avg_position_march,-0.0256,0.0256


In [47]:
# Generate ranked content recommendations from the test set

recommendations = model_data.loc[
    X_test.index,
    [
        "content_hash_id",
        "client_hash_id",
        "search_volume",
        "competition",
        "backlinks",
        "word_count",
        "avg_position_march",
        "future_clicks",
        "target"
    ]
].copy()

recommendations["model_score"] = model_scores

# Highest model score = highest priority
recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(1, len(recommendations) + 1)

print("Top 20 ranked recommendations:")
recommendations[
    [
        "rank",
        "content_hash_id",
        "model_score",
        "search_volume",
        "competition",
        "word_count",
        "avg_position_march"
    ]
].head(20)

Top 20 ranked recommendations:


,rank,content_hash_id,model_score,search_volume,competition,word_count,avg_position_march
0,1,content_d7db42380f907ad9,0.993140,140,0.77,10987,7.645723
1,2,content_ccd385b8a22ac069,0.990287,0,0.00,10280,32.117912
2,3,content_98123b029113fa39,0.978387,0,0.00,8818,23.791854
3,4,content_60a1acf8e356bd94,0.974754,0,0.00,8474,2.702543
4,5,content_05be4b02633ddd1f,0.973622,0,0.00,8476,30.330877
5,6,content_8336cb7cc972139d,0.971990,0,0.00,8277,NaN
6,7,content_135d1759617e8f2e,0.971765,0,0.00,8368,35.681646
7,8,content_ffb4e3a1c135f22e,0.971665,0,0.00,8280,8.122823
8,9,content_33afa7bded79c7d3,0.970273,0,0.00,8210,14.000791
9,10,content_ea7474d92d9701c3,0.969734,0,0.00,8242,35.907327


In [48]:
# Add simple, transparent reason codes to ranked recommendations

def get_reason(row):
    reasons = []

    if pd.notna(row["word_count"]) and row["word_count"] >= 8000:
        reasons.append("high_word_count")

    if pd.notna(row["search_volume"]) and row["search_volume"] >= 100:
        reasons.append("higher_search_volume")

    if pd.notna(row["competition"]) and row["competition"] <= 0.5:
        reasons.append("lower_competition")

    if pd.notna(row["avg_position_march"]) and row["avg_position_march"] <= 10:
        reasons.append("strong_march_position")

    if pd.notna(row["backlinks"]) and row["backlinks"] >= 10:
        reasons.append("more_backlinks")

    if not reasons:
        reasons.append("model_score_only")

    return ", ".join(reasons)


recommendations["reason_code"] = recommendations.apply(
    get_reason,
    axis=1
)

top20 = recommendations.head(20)

print("Top 20 recommendations with reason codes:")

top20[
    [
        "rank",
        "content_hash_id",
        "model_score",
        "reason_code"
    ]
]

Top 20 recommendations with reason codes:


,rank,content_hash_id,model_score,reason_code
0,1,content_d7db42380f907ad9,0.993140,"high_word_count, higher_search_volume, strong_..."
1,2,content_ccd385b8a22ac069,0.990287,"high_word_count, lower_competition"
2,3,content_98123b029113fa39,0.978387,"high_word_count, lower_competition"
3,4,content_60a1acf8e356bd94,0.974754,"high_word_count, lower_competition, strong_mar..."
4,5,content_05be4b02633ddd1f,0.973622,"high_word_count, lower_competition"
5,6,content_8336cb7cc972139d,0.971990,"high_word_count, lower_competition"
6,7,content_135d1759617e8f2e,0.971765,"high_word_count, lower_competition"
7,8,content_ffb4e3a1c135f22e,0.971665,"high_word_count, lower_competition, strong_mar..."
8,9,content_33afa7bded79c7d3,0.970273,"high_word_count, lower_competition"
9,10,content_ea7474d92d9701c3,0.969734,"high_word_count, lower_competition"


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [49]:
# ML-08 Section 2: Split design

from sklearn.model_selection import train_test_split

feature_names = [
    "search_volume",
    "competition",
    "backlinks",
    "word_count",
    "avg_position_march"
]

# Merge real features with future outcome
model_df = features_final.merge(
    future_outcome[
        [
            "content_hash_id",
            "client_hash_id",
            "future_clicks"
        ]
    ],
    on=[
        "content_hash_id",
        "client_hash_id"
    ],
    how="inner"
)

# Create target
# 1 = received at least one future click
# 0 = received zero future clicks
model_df["target"] = (
    model_df["future_clicks"] > 0
).astype(int)

# Select features and target
X = model_df[feature_names].copy()
y = model_df["target"].copy()

print("Model rows:", len(model_df))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget rate:")
print(y.mean())

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nSplit design:")
print("---------------------------")
print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

print("\nTraining positive rate:")
print(y_train.mean())

print("\nTesting positive rate:")
print(y_test.mean())

print("\nMissing values in training:")
print(X_train.isnull().sum().sum())

print("\nMissing values in testing:")
print(X_test.isnull().sum().sum())

Model rows: 324946

Target distribution:
target
0    262852
1     62094
Name: count, dtype: int64

Target rate:
0.1910902119121331

Split design:
---------------------------
Training rows: 259956
Testing rows : 64990

Training positive rate:
0.19109003062056656

Testing positive rate:
0.19109093706724112

Missing values in training:
0

Missing values in testing:
0


In [50]:
# ML-08 Section 2: Target sanity check

print("Target definition:")
print("Class 1 = future_clicks > 0")
print("Class 0 = future_clicks == 0")

print("\nFuture clicks:")
print(model_df["future_clicks"].describe())

print("\nTarget rate:", round(y.mean(), 4))

Target definition:
Class 1 = future_clicks > 0
Class 0 = future_clicks == 0

Future clicks:
count    324946.000000
mean          2.480658
std          21.901830
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        7434.000000
Name: future_clicks, dtype: float64

Target rate: 0.1911


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [51]:
# ML-08 Section 3: Train + compare vs baseline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# -----------------------------------------
# 1. Train Logistic Regression
# -----------------------------------------

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

# Probability scores
model_scores = logistic_model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Test predictions:", len(model_scores))


# -----------------------------------------
# 2. Baseline: March average-position ranking
# -----------------------------------------

baseline_scores = model_df.loc[
    X_test.index,
    "avg_position_march"
].copy()

# Lower position = better ranking
baseline_scores = -baseline_scores


# -----------------------------------------
# 3. Precision@K function
# -----------------------------------------

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_k_idx = np.argsort(scores)[::-1][:k]

    return y_true[top_k_idx].mean()


# -----------------------------------------
# 4. Calculate baseline metrics
# -----------------------------------------

baseline_p10 = precision_at_k(
    y_test,
    baseline_scores,
    10
)

baseline_p50 = precision_at_k(
    y_test,
    baseline_scores,
    50
)

baseline_p100 = precision_at_k(
    y_test,
    baseline_scores,
    100
)

baseline_pr_auc = average_precision_score(
    y_test,
    baseline_scores
)


# -----------------------------------------
# 5. Calculate ML model metrics
# -----------------------------------------

model_p10 = precision_at_k(
    y_test,
    model_scores,
    10
)

model_p50 = precision_at_k(
    y_test,
    model_scores,
    50
)

model_p100 = precision_at_k(
    y_test,
    model_scores,
    100
)

model_pr_auc = average_precision_score(
    y_test,
    model_scores
)


# -----------------------------------------
# 6. Comparison table
# -----------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Precision@50",
        "Precision@100",
        "PR-AUC"
    ],
    "Baseline": [
        baseline_p10,
        baseline_p50,
        baseline_p100,
        baseline_pr_auc
    ],
    "ML Model": [
        model_p10,
        model_p50,
        model_p100,
        model_pr_auc
    ]
})

comparison["Absolute improvement"] = (
    comparison["ML Model"] -
    comparison["Baseline"]
)

comparison["Relative improvement %"] = (
    comparison["Absolute improvement"] /
    comparison["Baseline"]
) * 100

comparison = comparison.round(4)

print("ML Ranking Model vs Baseline")
print("============================")

display(comparison)

Model trained successfully.
Test predictions: 64990
ML Ranking Model vs Baseline


,Metric,Baseline,ML Model,Absolute improvement,Relative improvement %
0,Precision@10,0.0000,0.5000,0.5000,inf
1,Precision@50,0.0200,0.3800,0.3600,1800.0000
2,Precision@100,0.0300,0.3700,0.3400,1133.3333
3,PR-AUC,0.3158,0.2806,-0.0352,-11.1317


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [52]:
# ML-08 Section 4: Errors and interpretation
# Ranking-focused error analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------------------
# 1. Create test ranking table
# -----------------------------------------

ranking_test = model_df.loc[
    X_test.index,
    [
        "content_hash_id",
        "client_hash_id",
        "future_clicks",
        "target",
        "search_volume",
        "competition",
        "backlinks",
        "word_count",
        "avg_position_march"
    ]
].copy()

ranking_test["model_score"] = model_scores
ranking_test["baseline_score"] = baseline_scores.values

# -----------------------------------------
# 2. Top 20 ML recommendations
# -----------------------------------------

top20_model = (
    ranking_test
    .sort_values("model_score", ascending=False)
    .head(20)
    .copy()
)

top20_model["rank"] = range(1, len(top20_model) + 1)

print("Top 20 ML-ranked recommendations")
print("=================================")

display(
    top20_model[
        [
            "rank",
            "content_hash_id",
            "model_score",
            "future_clicks",
            "search_volume",
            "competition",
            "word_count",
            "avg_position_march"
        ]
    ]
)

# -----------------------------------------
# 3. Top-K error analysis
# -----------------------------------------

for k in [10, 50, 100]:

    top_k = (
        ranking_test
        .sort_values("model_score", ascending=False)
        .head(k)
    )

    positives = int(top_k["target"].sum())
    precision = top_k["target"].mean()

    print(f"\nTop-{k} analysis")
    print("----------------")
    print("Relevant recommendations:", positives)
    print("Precision:", round(precision, 4))

# -----------------------------------------
# 4. Compare ML vs baseline PR-AUC
# -----------------------------------------

print("\nOverall ranking interpretation")
print("==============================")

print(
    f"ML Precision@10: {model_p10:.4f}"
)

print(
    f"ML Precision@50: {model_p50:.4f}"
)

print(
    f"ML Precision@100: {model_p100:.4f}"
)

print(
    f"ML PR-AUC: {model_pr_auc:.4f}"
)

print(
    f"Baseline PR-AUC: {baseline_pr_auc:.4f}"
)

if model_pr_auc > baseline_pr_auc:
    print(
        "\nThe ML model improves overall ranking quality according to PR-AUC."
    )
else:
    print(
        "\nThe ML model has lower overall PR-AUC than the baseline, "
        "despite stronger Precision@10, Precision@50 and Precision@100."
    )

# -----------------------------------------
# 5. Feature contribution
# -----------------------------------------

trained_model = logistic_model.named_steps["model"]

coefficients = trained_model.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute importance": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "Absolute importance",
    ascending=False
)

print("\nFeature contribution to the ranking model:")
print("==========================================")

display(
    feature_importance.round(4)
)

# -----------------------------------------
# 6. Strongest feature interpretation
# -----------------------------------------

strongest = feature_importance.iloc[0]

print("\nInterpretation")
print("-------------")

print(
    f"The strongest model coefficient is for "
    f"{strongest['Feature']}."
)

print(
    f"The coefficient is {strongest['Coefficient']:.4f}."
)

if strongest["Coefficient"] > 0:
    print(
        "Higher values of this feature are associated with "
        "higher predicted probability of future clicks, "
        "holding the other model features constant."
    )
else:
    print(
        "Higher values of this feature are associated with "
        "lower predicted probability of future clicks, "
        "holding the other model features constant."
    )

Top 20 ML-ranked recommendations


,rank,content_hash_id,model_score,future_clicks,search_volume,competition,word_count,avg_position_march
42583,1,content_d55a7c5a0f441d8c,0.858618,1.0,0,0.00,9031,8.984299
213110,2,content_d3f33b6020264644,0.838932,0.0,0,0.00,8705,8.511917
213011,3,content_5b6cb2a355ef0d7a,0.833391,0.0,110,0.27,8530,8.511917
44664,4,content_d030f704ff2f6d98,0.829258,0.0,0,0.00,8611,9.710822
153478,5,content_03e5f9aa591d5d11,0.828363,0.0,0,0.00,8301,1.990651
42109,6,content_5fe0aaf21402b5b0,0.815125,9.0,10,0.00,8179,3.502665
42449,7,content_6c0d323b0856b442,0.811927,2.0,0,0.00,8188,4.859375
42207,8,content_152a1aa206af56e5,0.811369,12.0,10,0.35,8310,11.682592
252015,9,content_e0e92e11b4839c90,0.809587,0.0,40,0.01,8298,8.511917
153068,10,content_f11cf5b5dfede22a,0.801613,24.0,10,0.00,8229,9.344782



Top-10 analysis
----------------
Relevant recommendations: 5
Precision: 0.5

Top-50 analysis
----------------
Relevant recommendations: 19
Precision: 0.38

Top-100 analysis
----------------
Relevant recommendations: 37
Precision: 0.37

Overall ranking interpretation
ML Precision@10: 0.5000
ML Precision@50: 0.3800
ML Precision@100: 0.3700
ML PR-AUC: 0.2806
Baseline PR-AUC: 0.3158

The ML model has lower overall PR-AUC than the baseline, despite stronger Precision@10, Precision@50 and Precision@100.

Feature contribution to the ranking model:


,Feature,Coefficient,Absolute importance
3,word_count,0.5071,0.5071
4,avg_position_march,-0.2600,0.2600
0,search_volume,-0.0880,0.0880
1,competition,0.0484,0.0484
2,backlinks,-0.0097,0.0097



Interpretation
-------------
The strongest model coefficient is for word_count.
The coefficient is 0.5071.
Higher values of this feature are associated with higher predicted probability of future clicks, holding the other model features constant.


In [53]:
# Section 4: Create class predictions for error analysis

# Get predicted probabilities from the trained ML model
model_prob = logistic_model.predict_proba(X_test)[:, 1]

# Convert probabilities to class predictions
# 0.5 is the standard classification threshold
model_pred = (model_prob >= 0.5).astype(int)

print("Class predictions created successfully.")
print("Test predictions:", len(model_pred))
print("Predicted class 0:", (model_pred == 0).sum())
print("Predicted class 1:", (model_pred == 1).sum())

Class predictions created successfully.
Test predictions: 64990
Predicted class 0: 64087
Predicted class 1: 903


In [54]:
# Section 4: Confusion Matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, model_pred)

print("Confusion Matrix")
print("================")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nError breakdown")
print("----------------")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

total_errors = fp + fn
error_rate = total_errors / len(y_test)

print("\nTotal incorrect predictions:", total_errors)
print("Error rate:", round(error_rate, 4))

Confusion Matrix
[[51970   601]
 [12117   302]]

Error breakdown
----------------
True Negatives : 51970
False Positives: 601
False Negatives: 12117
True Positives : 302

Total incorrect predictions: 12718
Error rate: 0.1957


In [55]:
# Section 4: Feature contribution / interpretation

trained_model = logistic_model.named_steps["model"]

coefficients = trained_model.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute_Importance": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "Absolute_Importance",
    ascending=False
)

print("Feature contribution to the ranking model:")
print("==========================================")

display(feature_importance.round(4))


# Strongest feature
strongest = feature_importance.iloc[0]

print("\nInterpretation")
print("-------------")

print(
    f"The strongest model coefficient is for {strongest['Feature']}."
)

print(
    f"The coefficient is {strongest['Coefficient']:.4f}."
)

if strongest["Coefficient"] > 0:
    print(
        "Higher values of this feature are associated with "
        "higher predicted probability of future clicks, "
        "holding the other model features constant."
    )
else:
    print(
        "Higher values of this feature are associated with "
        "lower predicted probability of future clicks, "
        "holding the other model features constant."
    )

Feature contribution to the ranking model:


,Feature,Coefficient,Absolute_Importance
3,word_count,0.5071,0.5071
4,avg_position_march,-0.2600,0.2600
0,search_volume,-0.0880,0.0880
1,competition,0.0484,0.0484
2,backlinks,-0.0097,0.0097



Interpretation
-------------
The strongest model coefficient is for word_count.
The coefficient is 0.5071.
Higher values of this feature are associated with higher predicted probability of future clicks, holding the other model features constant.


### Section 4: Errors and Interpretation

The model shows stronger precision among the highest-ranked recommendations, with Precision@10 of 0.50, Precision@50 of 0.38, and Precision@100 of 0.37. However, its PR-AUC of 0.2806 is lower than the baseline PR-AUC of 0.3158, so the model should not be described as an overall improvement.

The confusion matrix shows 51,970 true negatives, 601 false positives, 12,117 false negatives, and 302 true positives. The relatively high number of false negatives indicates that the classification threshold misses many actual positive cases.

For feature interpretation, word_count has the largest absolute coefficient (0.5071), making it the strongest feature in this fitted logistic model. Its positive coefficient indicates that higher word count is associated with higher predicted probability of future clicks, holding the other features constant. The coefficients are directional associations from the fitted model and should not be interpreted as causal effects.

Overall, the model appears useful for prioritizing a smaller set of high-ranking recommendations, but its lower PR-AUC means the ranking model should be treated as decision-support rather than a definitive predictor.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.